### LangChain 필요성
- OpenAI API 직접 사용 시
    - 사용자 질문 -> LLM 호출 -> 응답
- 실제 서비스에서 필요한 기능
    - PDF 문서 안에서 답 찾기
    - DB 조회 후 결과 설명
    - 이전 대화 기억
    - 여러 단계 추룬

### LangChain 핵심 개념
- Chain : 여러 단계를 연결한 처리 흐름
    - ex) 검색 + 요약 + 생성 같은 다단계 흐름 구성 가능
- Prompt Template : 프롬프트를 템플릿화
- Memory : 이전 대화 저장
- Retriever(검색기)
- Agent : LLM이 스스로 도구를 선택하여 실행

#### 활용 가능 분야
- 문서기반 QA
    - ex) 사내문서 QA 챗봇, PDF 기반 질의응답, 법률 문서 검색, 기술 매뉴얼 검색
- 챗봇 시스템
- 데이터 분석 AI
- 에이전트 기반 자동화 시스템

### RAG(Retrieval Augmented Generation)
- 검색 증강 생성
- 특정 자료(문서,PDF,DB 등)에 대해 질문에 답할 수 있다.
- 2가지 방식 제공
    - RAG Agent 방식 : Agent를 이용해 여러 번 검색이 가능하고 복잡한 질문 처리 가능
    - Two-step RAG Chain : 단순한 형태 / 한 번 호출
- 처리 단계
    - 1. Indexing : 데이터를 어떤 소스로부터 가져와서 검색할 수 있도록

In [ ]:
# !pip install langchain_openai

   ---------------------------------------- 0.0/921.1 kB ? eta -:--:--
   ---------------------------------------- 921.1/921.1 kB 15.4 MB/s  0:00:00

   ---------------------------------------- 0/3 [regex]
   -------------------------- ------------- 2/3 [langchain_openai]
   -------------------------- ------------- 2/3 [langchain_openai]
   ---------------------------------------- 3/3 [langchain_openai]



In [36]:
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
import bs4
from pathlib import Path

load_dotenv(find_dotenv())

True

In [37]:
# 랭체인에서 제공하는 로더 
from langchain_community.document_loaders import YoutubeLoader, WebBaseLoader, PyPDFLoader 

# 임베딩 처리
from langchain_community.vectorstores import FAISS

# 문서 내용 임베딩
from langchain_openai import OpenAIEmbeddings
# 모델 생성
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model

# 메세지
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# 프롬프트 작성
from langchain_core.prompts import PromptTemplate 

# 글을 쪼갤 때 사용하는 라이브러리
from langchain_text_splitters  import RecursiveCharacterTextSplitter

# Tools
from langchain.tools import tool
from langchain.agents import create_agent

# 크롤링
from langchain_community.document_loaders.firecrawl import FireCrawlLoader

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [4]:
# 모델 생성
# 방법 1
# model = init_chat_model(model="gpt-5-nano", temperature=0, max_token=500, timeout=60)

# 방법 2
model = ChatOpenAI(model="gpt-5-nano")

##### Langchain
- 모델 호출
    - 1) invoke() : 모델이 답변을 전체 생성 후 응답
    - 2) stream() : 모델이 생성한 답변을 중간중간 응답(논문요약, 보고서 생성, 코드 생성, 긴 설명)
    - 3) batch() : 여러 개 요청을 한 번에 처리(병렬 처리 가능 - 리뷰 100 개 요약, 문단 50개 번역..)

In [5]:
# 모델 호출 : invoke()
# 하나의 메시지 입력 시

res = model.invoke("과민성대장증후군 해결법을 설명해줘")
print(res)
print(res.usage_metadata)
print(res.content)

content='과민성대장증후군(IBS)은 대장 기능의 일시적 이상으로 복통이나 복부 불편감과 함께 배변 습관이 자주 변하는 만성 질환이에요. 장의 구조적 문제가 아니라 기능적 문제로 생각되며, 스트레스나 식사 습관, 장-뇌 축 등의 영향으로 증상이 악화되곤 합니다. 다행히 관리하면 증상을 크게 개선할 수 있습니다.\n\nIBS의 주요 유형\n- IBS-C: 변비가 주가 되며 배변 후에도 불편감이 남는 경우\n- IBS-D: 설사가 주가 되며 급하고 잦은 배변이 특징\n- IBS-M: 변비와 설사가 번갈아 나타남\n- IBS-U: 다른 유형으로 분류되지 않는 경우\n\n진단 및 주의점\n- IBS는 보통 증상과 배변 습관의 변화로 진단되지만, 혈변, 체중감소, 지속적인 발열, 매우 젖은 피로감 등 red flags가 있으면 다른 질환 여부를 확인하기 위해 추가 검사를 합니다.\n- 의사와 상의해 보시고, 필요시 혈액검사, 대변검사, celiac 질환 검사, 내시경 검사 등을 진행합니다.\n- 자가진단 대신 전문가와 증상 기록(배변 모양, 통증 위치/강도, 시간)을 공유하는 것이 도움이 됩니다.\n\n증상 관리: 시작하기 좋은 전략\n1) 식이(weight 관리 포함)\n- 저 FODMAP 식단: 특정 당류를 줄이면 증상이 완화될 수 있어요. 다만 기간은 약 4~6주가 적당하고, 잦은 영양소 결핍을 피하려면 점진적으로 진행하며 전문가의 지도를 받는 것이 좋습니다.\n- 수용성 섬유질(예: 차전자섬유 psyllium)은 IBS-C에 도움될 수 있고, IBS-D에서는 불편감을 악화시킬 수 있어 개인 차를 보며 적용합니다.\n- 물 충분히 마시기, 규칙적인 식사. 카페인, 알코올, 매운 음식, 기름진 음식은 증상을 자극할 수 있어 개인 차를 관찰하며 줄이는 게 좋습니다.\n- 특정 음식을 트리거로 느끼면 식이일지를 통해 원인을 찾는 것이 좋습니다.\n2) 생활습관\n- 규칙적인 운동과 충분한 수면은 장운동과 스트레스 관리에 도움이 됩니다.\n- 스트레스 관리: 명상,

In [6]:
# stream() : 모델이 생성한 답변을 중간중간 응답
agent = create_agent(model)

for step in agent.stream(
    {"messages": [{"role": "user", "content": "과민성대장증후군 해결법을 설명해줘"}]},
    stream_mode="values", # values : 현재 상태 전체, updates : update 내용만
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

과민성대장증후군 해결법을 설명해줘
================================== Ai Message ==================================

과민성대장증후군(IBS)은 생명을 위협하지 않는 기능성 소화기 질환으로, 증상을 잘 관리하는 것이 핵심이에요. 개인에 따라 힘들어하는 증상(배변 습관 변화, 복통, 팽만감 등)이 다 다르기 때문에 한 가지 해답보다 본인에게 맞춘 복합 관리가 필요합니다. 아래를 참고해 보시고, 필요하면 의사와 상의해 맞춤 계획을 세우세요.

1) 진단과 기본 원칙
- IBS는 보통 배변 패턴과 복통의 반복으로 진단합니다(일정 기간의 증상 지속 및 다른 질환 배제).
- 악화되는 증상이나 체중 감소, 피혈변,持续적인 구토, 발열 등 ‘적신호’가 있으면 즉시 의사 상담이 필요합니다.
- 스스로의 상태를 정확히 기록해 두면(증상 시작 시점, 식사와의 연관, 스트레스 등) 진단과 관리에 도움이 됩니다.

2) 생활습관 관리
- 규칙적인 식사와 충분한 수분 섭취
- 규칙적 운동은 증상 완화에 도움
- 스트레스 관리: 명상, 호흡법, 수면 관리, 심리적 스트레스가 증상을 악화시킬 수 있어요.
- 자극 음식 피하기: 카페인, 알코올, 매운 음식, 아주 기름진 음식 등 일부 사람에게 증상을 악화시킬 수 있습니다.
- 식사 기록과 증상 다이어리 활용: 어떤 음식이 증상에 영향을 주는지 확인합니다.

3) 식이 조정(저 FODMAP 다이어트 등)
- 저 FODMAP 식이: 특정 당류를 줄이면 증상이 완화될 수 있습니다. 초기에는 제한이 까다로울 수 있어 전문 영양사와 함께 진행하는 것이 좋습니다.
- 점성 섬유(수용성 섬유) 섭취: psyllium 같은 섬유질은 변비가 있을 때 증상을 완화할 수 있습니다. 과도하게 갑자기 먹기보다 천천히 시작해 용량을 올리세요.
- 개인 차

In [7]:
# batch()
inputs = [
    "사과의 효능을 설명해줘",
    "과민성대장증후군 해결법을 설명해줘",
    "langchain에 대해 설명해줘",
]

res = model.batch(inputs)

for r in res:
    print(r.content)

사과는 영양가가 높은 과일로, 다양한 건강 효과를 기대할 수 있습니다.

주요 영양 성분(중간 크기 한 개 기준)
- 칼로리 약 95 kcal, 식이섬유 약 4 g
- 비타민 C 약 8 mg, 칼륨 약 195 mg
- 당류는 자연당(과당, 포도당)으로 포함
- 껍질에 폴리페놀(퀘르세틴, 카테킨, 클로로제닉 산 등)과 같은 항산화 물질이 많습니다

사과의 건강 효능
- 소화 건강과 포만감: 식이섬유(특히 펙틴)가 장 운동을 도와 변비 예방에 도움을 줄 수 있고 포만감을 주어 식사 조절에 도움이 됩니다.
- 심혈관 건강: 섬유질과 항산화 물질이 혈중 나쁜 LDL 콜레스테롤 감소, 혈압 관리, 혈관 건강에 긍정적 영향을 줄 수 있습니다.
- 혈당 관리: 식이섬유와 폴리페놀의 작용으로 탄수화물 흡수를 느리게 하고 혈당 상승을 완만하게 도울 수 있어 당뇨 예방/관리에 도움이 될 수 있습니다.
- 항산화 및 염증 감소: 껍질에 많은 퀘르세틴과 다른 폴리페놀이 체내 산화 스트레스와 염증을 감소시키는 데 기여할 수 있습니다.
- 체중 관리: 칼로리는 비교적 낮고 섬유질이 포만감을 높여 과식 예방에 도움이 될 수 있습니다.
- 구강 건강: 씹는 과정에서 타액 분비가 증가해 구강 청결에 간접적으로 이로울 수 있습니다.
- 뇌 건강 및 노화 관련 분야: 폴리페놀이 뇌 기능 보호에 도움이 될 수 있다는 연구도 있지만, 아직은 추가 증거가 필요합니다.

섭취 팁
- 가능하면 껍질째 먹기: 껍질에 많은 섬유질과 항산화 물질이 들어 있습니다.
- 신선하고 씻어서 섭취: 살균된 상태로 먹는 것이 좋고, 유기농 여부도 선택 포인트가 될 수 있습니다.
- 주스는 주의: 과일 주스는 섬유질이 줄고 당이 빨리 흡수될 수 있어, 과일 한 개 분량 정도를 천천히 마시거나 가급적 통째로 먹는 것이 좋습니다.
- 보관: 냉장 보관으로 신선도를 오래 유지합니다.
- 주의할 점: 당뇨가 있거나 당 섭취를 관리해야 하는 경우에는 섭취량을 조절하고, 과다 섭취는 피하는 것이 좋습니다.

필요하다면 특정 상황(

- Messages
    - https://docs.langchain.com/oss/python/langchain/messages

In [8]:
system_msg = SystemMessage("You are a helpful assistant.")
human_msg = HumanMessage("Hello, how are you?")

response = model.invoke([system_msg, human_msg])

print(response.content)

I'm doing well, thank you! How about you? What can I help you with today?


In [9]:
# 여러 개의 메시지 리스트
# https://docs.langchain.com/oss/python/langchain/models#invoke
# 대화기록
messages = [
    {
        "role": "user",
        "content": "2002년 월드컵에서 가장 화제가 되었던 나라는 어디야?",
    },
    {
        "role": "assistant",
        "content": "대한민국이 화제가 됨",
    },
    {
        "role": "user",
        "content": "그 나라가 화제가 됐던 이유를 자세히, 하지만 팩트로만 설명해줘.",
    },
]

response = model.invoke(messages)
print(response.content)

다음은 팩트 위주로 정리한 주요 내용입니다.

- 주최: 2002년 월드컵은 한국과 일본이 공동 개최되었다. 이는 아시아에서 최초의 공동 개최였다.
- 역사적 성과: 한국은 월드컵 역사상 최초로 아시아 팀으로 4강에 진출했다.
- 16강: 이탈리아를 2-1로 이겼다. 결정적 골은 연장전 골든골로 나왔다(117분).
- 8강: 스페인을 승부차기로 이겼다. 양 팀은 0-0으로 종료된 뒤 승부차기에서 한국이 승리했다.
- 준결승: 독일에 0-1로 패배했다.
- 3위전: 터키에 3-2로 패배하며 4위를 기록했다.
- 최종 결과: 브라질이 우승하고 독일이 준우승을 차지했다.
- 대회의 영향: 아시아 출신 팀의 월드컵 성과가 글로벌 이목을 크게 끌었고, 이후 아시아 축구의 관심과 투자가 확대되는 계기가 되었다.


- Tools : 앞에서 했었던 function calling 
    - https://docs.langchain.com/oss/python/langchain/tools

In [ ]:
# 앞에서 했었던 function calling
import json
import requests


@tool
def get_weather(location, timezone):
    """지역명을 이용해 위,경도를 찾은 후 해당 날씨 가져오기"""
    pass


tools = [get_weather]

agent = create_agent(model=model, tools=tools)
agent.invoke({"message": [{"role": "user", "content": "서울 날씨 어때"}]})

res["messages"][-1].pretty_print()

### LangChain 을 사용한 RAG
#### [실습 1] blog 를 기반으로 질의 검색

In [12]:
# Load and chunk contents of the blog
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    header_template={"User-Agent": "my-langchain-rag-app"},
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# print(docs)
# 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)


# 검색
# 단계 1. 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-samll")
# 단계 2. 문서의 벡터화, 벡터DB에 저장
vector_store = FAISS.from_documents(all_splits, embeddings)


@tool
def retrieve_context(query):
    """사용자 질문을 받아 벡터 검색 수행 / 관련문서 2개 가져온 후 텍스트 형태로 변환"""
    # query에 대한 임베딩 생성 자동
    retrieve_docs = vector_store.similarity_search(query=query, k=2)
    serialized = "\n\n".join(
        (f"Source : {doc.metadata}\nContent : {doc.page_content}")
        for doc in retrieve_docs
    )

    return serialized, retrieve_docs


tools = [retrieve_context]
prompt = """
You have a access to a tool that retrieves context from a blog post.
Use the tool to help answer user queries.
"""

agent = create_agent(model, tools, system_prompt=prompt)

query = "What is task decomposition?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]}, stream_mode="values"
):
    step["messages"][-1].pretty_print()

NotFoundError: Error code: 404 - {'error': {'message': 'The model `text-embedding-3-samll` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

#### [실습 2] PDF 기반 질의탐색

In [13]:
# 1. 문서 로드
pdf_path = "./data/Summary of ChatGPTGPT-4 Research.pdf"

if not Path(pdf_path).exists:
    raise FileNotFoundError()

loader = PyPDFLoader(pdf_path)
docs = loader.load()

In [14]:
# 2. 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

122

In [16]:
print(all_splits[0])

page_content='Summary of ChatGPT/GPT-4 Research
and Perspective Towards the Future of Large
Language Models
Yiheng Liu ∗1, Tianle Han ∗1, Siyuan Ma 1, Jiayue Zhang 1,
Yuanyuan Yang1, Jiaming Tian 1, Hao He 1, Antong Li 2, Mengshen
He1, Zhengliang Liu 3, Zihao Wu 3, Dajiang Zhu 4, Xiang Li 5, Ning
Qiang1, Dingang Shen 6,7,8, Tianming Liu 3, and Bao Ge †1
1School of Physics and Information Technology, Shaanxi Normal University, Xi’an
710119 China
2School of Life and Technology Biomedical-Engineering, Xi’an Jiaotong University,
Xi’an 710049, China
3School of Computing, The University of Georgia, Athens 30602, USA
4Department of Computer Science and Engineering, The University of Texas at
Arlington, Arlington 76019, USA
5Department of Radiology, Massachusetts General Hospital and Harvard Medical
School, Boston 02115, USA
6School of Biomedical Engineering, ShanghaiTech University, Shanghai 201210,
China
7Shanghai United Imaging Intelligence Co., Ltd., Shanghai 200230, China' metadata={'prod

In [17]:
# 3. 백터 스토어 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(documents=all_splits, embedding=embeddings)

In [19]:
# 4. 검색 tool 작성
@tool
def retrieve_context(query):
    "사용자 질문을 받아 벡터 검색 수행 / 관련문서 k개 가져온 후 텍스트 형태로 반환"

    retrieve_docs = vector_store.similarity_search(query=query, k=4)
    serialized = "\n\n".join(
        (f"Source : {doc.metadata}\nContent : {doc.page_content}")
        for doc in retrieve_docs
    )

    return serialized, retrieve_docs


tools = [retrieve_context]

In [20]:
# 5. 모델 생성
prompt = """
You have a access to a tool that retrieves context from a pdf document.
Use the tool when you need evidence form the PDF.
"""
agent = create_agent(model, tools, system_prompt=prompt)

In [21]:
# 6. 질문 던지기

query = "What can i use chatGPT"

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]}, stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What can i use chatGPT
================================== Ai Message ==================================

ChatGPT is a conversational AI you can use for a wide range of tasks. Here are common ways people rely on it:

- Writing and editing
  - Drafting emails, reports, resumes, essays
  - Polishing style, tone, and grammar
  - Generating summaries or paraphrasing

- Learning and explanations
  - Breaking down complex concepts (math, science, history)
  - Step-by-step problem solving
  - Language practice and explanations in simple terms

- Coding and technical help
  - Debugging code, explaining bugs, writing snippets
  - Generating algorithms, pseudocode, or scripts
  - Translating technical ideas into plain language

- Planning and organization
  - Creating to-do lists, study plans, travel itineraries
  - Structuring projects, timelines, and outlines
  - Brainstorming next steps and milestones

- Research

#### [실습 3] 유튜브 영상 자막 추출 후 내용요약

- YoutubeLoader 를 이용한 자막 추출
- RecursiveCharacterTextSplitter 를 이용한 일정한 크기로 자막 분할
- agent 를 이용한 LLM 호출 
- 응답

In [28]:
# 1. 유투브 자막 로드
loader = YoutubeLoader.from_youtube_url("https://www.youtube.com/watch?v=ZXRD_-jFIM4")
transcript = loader.load()

In [29]:
# 2. 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(transcript)

In [30]:
# 3. Map 단계 (각 chunk 요약)
map_prompt = ChatPromptTemplate.from_template("""
Summarize the follow transcript chunk:
```{text}```
                                              
Chunk summary
"""
)

map_chain = map_prompt | model | StrOutputParser()

# chunk 요약
chunk_summaries = []
for doc in all_splits:
    summary = map_chain.invoke({"text": doc.page_content})
    chunk_summaries.append(summary)

In [31]:
# 4. Reduce 단계 (chunk 결합 요약)
combine_prompt = ChatPromptTemplate.from_template(
    """
You are given partial summaries of a Youtube's transcript.
Combine them into a fianl Korean.
- 8 ~ 10 문장
- 핵심 문장 / 근거 / 예시 / 결론 포함
                                                  
Summary
```{text}```
                                                  
Final summary (Korean)
"""
)

combine_chain = combine_prompt | model | StrOutputParser()

final_summary = combine_chain.invoke({"text": "\n\n".join(chunk_summaries)})
print(final_summary)

연설자는 경제 성장, 안보 강화, 가족 지원, 그리고 우리 나라의 품성을 실천하자는 목표를 제시한다. 그는 특히 젊은이들의 열정과 우려를 인정하며 공감의 메시지를 전한다. 문제를 법의 테두리 안에서 해결하겠다는 의지를 분명히 밝힌다. 의회가 존재하는 한 가능하다면 의회를 우회하지 않겠다는 원칙을 강조한다. 미국은 법의 나라이기에 불법적 수단으로 해법을 찾는 길은 옳지 않다고 말한다. 이민 개혁은 민주적 절차를 통해서만 달성되어야 한다고 촉구하고, 로비와 행진, 조정된 노력 등 다양한 합법적 방법을 예로 든다. 그는 열심히 일하는 이민자들을 지지하며, 그들이 공유하는 미국의 가치관을 환영하겠다고 약속한다. 시스템 내에서의 노력이야말로 성공의 길이며, 법을 어겨 얻는 것은 아니라고 강조한다. 민주주의가 때로 혼란스럽고 힘들 수 있지만, 정의와 진실은 역사를 통해 이길 것이며 오늘도 그 원칙은 지켜질 것이라고 결론짓는다.


#### Firecrawl
- 웹사이트를 크롤링하여 LLM(Long Library Management)에서 사용할 수 있는 데이터로 변환
- 접근 가능한 모든 하위 페이지를 크롤링하고 각 페이지에 대한 깔끔한 마크다운과 메타데이터를 제공
- https://docs.langchain.com/oss/python/integrations/document_loaders/firecrawl (api 키 발급 필요)

In [38]:
from langchain_community.document_loaders.firecrawl import FireCrawlLoader

In [39]:
# mode
# scrape : 지정한 url 페이지 하나
# crawl : 하위 페이지까지 내용 크롤링
loader = FireCrawlLoader(url="https://firecrawl.dev", mode="crawl")

docs = loader.load()

print(len(docs))

loader = FireCrawlLoader(url="https://naver.com", mode="crawl", params={"limit": 5})
docs = loader.load()
print(len(docs))

KeyboardInterrupt: 

#### [실습] react 페이지 크롤링 후 크롤링 문서 기반으로 RAG 적용
- 1. Ingest(색인) 단계: URL들 → Firecrawl → Document → Split → Embedding → FAISS 저장
- 2. Query(질의) 단계: 질문 → FAISS 검색(top-k) → (질문 + 근거) → LLM 답변

In [40]:
import re

loader = FireCrawlLoader(
    url="https://ko.react.dev/reference/react/",
    mode="crawl",
    params={
        "includePaths": [r"^/reference/react/use.*"],
        "limit": 100,  # 넓게 크롤링
    },
)

docs = loader.load()

print(len(docs))


def get_url(doc):
    md = doc.metadata or {}
    return md.get("source_url") or md.get("source") or md.get("url") or "(unknown)"


# use~ 시작하는 url 만 걸러내기
pattern = re.compile(r"^https://ko\.react\.dev/reference/react/use.*")
use_docs = [d for d in docs if pattern.match(get_url(d))]
print(len(use_docs))

KeyboardInterrupt: 

In [ ]:
# 벡터 작업
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(use_docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(documents=all_splits, embedding=embeddings)


@tool(response_format="content_and_artifact")
def retrieve_context(query):
    "url로 수집된 정보를 기반으로 질문을 받아 벡터 검색 수행 / 관련문서 k개 가져온 후 텍스트 형태로 반환"

    retrieve_docs = vector_store.similarity_search(query=query, k=4)
    serialized = "\n\n".join(
        (f"Source : {get_url(doc)}\nContent : {doc.page_content}")
        for doc in retrieve_docs
    )

    return serialized, retrieve_docs


tools = [retrieve_context]

prompt = """
너는 React 공식 문서(한국어)를 기반으로 답변하는 도우미다.
공식 문서 속 정보로만 대답하고, 모르면 모른다고 말하라. 거짓말을 하지말아라.
답변은 한국어로, 보기 좋게 문서형식으로 정리하고 마지막에 출처를 남겨라.
"""
agent = create_agent(model, tools, system_prompt=prompt)

query = "useMemo와 useCallback의 차이점과 활용 예제에 대해 알려주세요."

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]}, stream_mode="values"
):
    step["messages"][-1].pretty_print()